In [1]:
!pip install -q openai

In [ ]:

import os
from getpass import getpass
from openai import OpenAI

api_key = getpass("Enter your OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("OpenAI client initialized successfully!")

In [ ]:

EVALUATION_CRITERIA = {
    "groundedness": {
        "description": """
        The answer must be supported ONLY by the supplied retrieved context.
        The assistant must not fabricate facts, sources, numbers, names,
        or information that is not present in the context.
        """,
        "scale": {
            1: "Completely unsupported or mostly fabricated",
            2: "Many unsupported claims",
            3: "Partially grounded with some unsupported claims",
            4: "Mostly grounded with minor unsupported details",
            5: "Fully supported by the provided context"
        }
    },

    "correctness": {
        "description": """
        The answer must match the manually written ground-truth answer.
        Minor wording differences are acceptable if the meaning is correct.
        """,
        "scale": {
            1: "Completely incorrect",
            2: "Mostly incorrect",
            3: "Partially correct",
            4: "Mostly correct with minor issues",
            5: "Fully correct"
        }
    },

    "completeness": {
        "description": """
        The answer must address all important parts of the question.
        Important information from the ground truth should not be omitted.
        """,
        "scale": {
            1: "Almost completely incomplete",
            2: "Major information missing",
            3: "Some important information missing",
            4: "Nearly complete",
            5: "Fully complete"
        }
    }
}

print("Evaluation criteria defined.")

In [ ]:

evaluation_dataset = [

    {
        "id": 1,
        "question": "What is artificial intelligence?",
        "context": """
        Artificial intelligence is the field of building computer systems
        that can perform tasks that normally require human intelligence,
        such as reasoning, learning, and decision making.
        """,
        "ground_truth": """
        Artificial intelligence is the field of building computer systems
        capable of performing tasks that normally require human intelligence,
        including reasoning, learning, and decision making.
        """
    },

    {
        "id": 2,
        "question": "What is machine learning?",
        "context": """
        Machine learning is a branch of artificial intelligence in which
        systems learn patterns from data and use those patterns to make
        predictions or decisions without being explicitly programmed
        for every individual task.
        """,
        "ground_truth": """
        Machine learning is a branch of AI where systems learn patterns
        from data to make predictions or decisions without being explicitly
        programmed for every task.
        """
    },

    {
        "id": 3,
        "question": "What is deep learning?",
        "context": """
        Deep learning is a subset of machine learning that uses neural
        networks with multiple layers to learn complex patterns from data.
        """,
        "ground_truth": """
        Deep learning is a subset of machine learning that uses
        multi-layer neural networks to learn complex patterns from data.
        """
    },

    {
        "id": 4,
        "question": "What is an artificial neural network?",
        "context": """
        An artificial neural network is a computational model inspired
        by biological neural networks. It consists of interconnected
        nodes called neurons organized into layers.
        """,
        "ground_truth": """
        An artificial neural network is a computational model inspired
        by biological neural networks, consisting of interconnected
        neurons organized into layers.
        """
    },

    {
        "id": 5,
        "question": "What is NLP?",
        "context": """
        Natural Language Processing, or NLP, is a field of AI that enables
        computers to process, understand, and generate human language.
        """,
        "ground_truth": """
        NLP is a field of AI that enables computers to process,
        understand, and generate human language.
        """
    },

    {
        "id": 6,
        "question": "What is tokenization?",
        "context": """
        Tokenization is the process of breaking text into smaller units
        called tokens. Tokens can be words, subwords, or characters.
        """,
        "ground_truth": """
        Tokenization breaks text into smaller units called tokens,
        which can be words, subwords, or characters.
        """
    },

    {
        "id": 7,
        "question": "What is an embedding?",
        "context": """
        An embedding is a numerical vector representation of data such
        as text. Embeddings represent semantic meaning so that similar
        pieces of information can have similar vector representations.
        """,
        "ground_truth": """
        An embedding is a numerical vector representation that captures
        semantic meaning, allowing similar information to have similar
        vector representations.
        """
    },

    {
        "id": 8,
        "question": "What is semantic search?",
        "context": """
        Semantic search retrieves information based on meaning rather
        than only matching exact keywords. Embeddings can be used to
        compare the semantic similarity between a query and documents.
        """,
        "ground_truth": """
        Semantic search retrieves information based on meaning rather
        than exact keyword matching, often using embeddings to compare
        semantic similarity.
        """
    },

    {
        "id": 9,
        "question": "What is RAG?",
        "context": """
        Retrieval-Augmented Generation, or RAG, combines information
        retrieval with text generation. Relevant documents are retrieved
        first and then supplied to a language model to generate an answer.
        """,
        "ground_truth": """
        RAG combines retrieval and generation. Relevant documents are
        retrieved first and then provided to a language model to generate
        an answer.
        """
    },

    {
        "id": 10,
        "question": "Why is retrieval useful in RAG?",
        "context": """
        Retrieval provides the language model with relevant external
        information that can be used to answer a question. This can
        reduce reliance on information stored only in the model's parameters.
        """,
        "ground_truth": """
        Retrieval provides relevant external information to the language
        model, reducing its reliance on information stored only in its
        model parameters.
        """
    },

    {
        "id": 11,
        "question": "What is a vector database?",
        "context": """
        A vector database stores numerical vector representations and
        supports similarity searches over those vectors. It is commonly
        used for semantic search and retrieval systems.
        """,
        "ground_truth": """
        A vector database stores numerical vector representations and
        supports similarity search. It is commonly used for semantic
        search and retrieval systems.
        """
    },

    {
        "id": 12,
        "question": "What is FAISS?",
        "context": """
        FAISS is a library for efficient similarity search and clustering
        of dense vectors. It is commonly used to build vector search
        systems.
        """,
        "ground_truth": """
        FAISS is a library for efficient similarity search and clustering
        of dense vectors and is commonly used for vector search systems.
        """
    },

    {
        "id": 13,
        "question": "What is prompt engineering?",
        "context": """
        Prompt engineering is the practice of designing and refining
        instructions given to language models to obtain better and more
        reliable outputs.
        """,
        "ground_truth": """
        Prompt engineering is the practice of designing and refining
        instructions for language models to improve the quality and
        reliability of their outputs.
        """
    },

    {
        "id": 14,
        "question": "What is an LLM?",
        "context": """
        A large language model is a machine learning model trained on
        large amounts of text data to understand and generate human-like
        language.
        """,
        "ground_truth": """
        An LLM is a machine learning model trained on large amounts of
        text data to understand and generate human-like language.
        """
    },

    {
        "id": 15,
        "question": "What is hallucination in an LLM?",
        "context": """
        Hallucination occurs when a language model produces information
        that is false, unsupported, or not present in the available
        source material.
        """,
        "ground_truth": """
        An LLM hallucination occurs when the model generates false,
        unsupported, or source-inconsistent information.
        """
    },

    {
        "id": 16,
        "question": "Why are evaluation metrics important for AI systems?",
        "context": """
        Evaluation metrics provide objective measurements of system
        performance. They allow developers to compare versions, detect
        regressions, and improve systems using data rather than intuition.
        """,
        "ground_truth": """
        Evaluation metrics objectively measure AI system performance,
        helping developers compare versions, detect regressions, and
        improve systems based on data rather than intuition.
        """
    },

    {
        "id": 17,
        "question": "What is regression testing for an AI system?",
        "context": """
        Regression testing checks whether changes to an AI system cause
        previously working behavior to become worse. A fixed evaluation
        dataset can be run after changes to detect performance drops.
        """,
        "ground_truth": """
        Regression testing checks whether system changes cause previously
        working behavior to become worse by rerunning a fixed evaluation
        dataset and detecting performance drops.
        """
    },

    {
        "id": 18,
        "question": "What is LLM-as-a-judge?",
        "context": """
        LLM-as-a-judge uses one language model to evaluate the output
        produced by another language model according to defined criteria.
        """,
        "ground_truth": """
        LLM-as-a-judge uses a language model to evaluate another model's
        output according to predefined evaluation criteria.
        """
    },

    {
        "id": 19,
        "question": "What is groundedness?",
        "context": """
        Groundedness measures whether an answer is supported by the
        provided context and avoids unsupported or fabricated information.
        """,
        "ground_truth": """
        Groundedness measures whether an answer is supported by the
        provided context without unsupported or fabricated information.
        """
    },

    {
        "id": 20,
        "question": "What is completeness?",
        "context": """
        Completeness measures whether an answer addresses all important
        parts of the user's question without omitting necessary information.
        """,
        "ground_truth": """
        Completeness measures whether an answer addresses all important
        parts of the question without omitting necessary information.
        """
    }
]

print("Dataset size:", len(evaluation_dataset))

In [ ]:

def knowledge_assistant(question, context):
    prompt = f"""
You are a knowledge assistant.

Answer the question using ONLY the supplied context.

Do not add information that is not present in the context.
If the context does not contain enough information, say so.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a precise knowledge assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [ ]:

import json

def llm_judge(question, context, answer, ground_truth):

    judge_prompt = f"""
You are an expert AI evaluation judge.

Evaluate the assistant answer on exactly THREE dimensions:

1. GROUNDEDNESS
Definition:
The answer must be supported ONLY by the provided context.
Penalize fabricated or unsupported information.

2. CORRECTNESS
Definition:
The answer must match the ground-truth answer in meaning.
Minor wording differences are acceptable.

3. COMPLETENESS
Definition:
The answer must answer all important parts of the question.
Penalize important omissions.

Use a score from 1 to 5.

Scoring:
1 = Very poor
2 = Poor
3 = Average / partially acceptable
4 = Good
5 = Excellent

QUESTION:
{question}

CONTEXT:
{context}

ASSISTANT ANSWER:
{answer}

GROUND TRUTH:
{ground_truth}

Return ONLY valid JSON.

Required format:

{{
    "groundedness": <integer 1-5>,
    "correctness": <integer 1-5>,
    "completeness": <integer 1-5>,
    "reasoning": {{
        "groundedness": "<short explanation>",
        "correctness": "<short explanation>",
        "completeness": "<short explanation>"
    }}
}}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a strict and objective AI evaluator."
            },
            {
                "role": "user",
                "content": judge_prompt
            }
        ],
        temperature=0
    )

    raw_output = response.choices[0].message.content.strip()

    try:
        result = json.loads(raw_output)

        # Validate scores
        for dimension in [
            "groundedness",
            "correctness",
            "completeness"
        ]:
            score = result[dimension]

            if not isinstance(score, int) or not 1 <= score <= 5:
                raise ValueError(
                    f"Invalid {dimension} score: {score}"
                )

        return result

    except Exception as e:
        print("Judge parsing error:", e)
        print("Raw judge output:", raw_output)

        return {
            "groundedness": 0,
            "correctness": 0,
            "completeness": 0,
            "reasoning": {
                "groundedness": "Parsing error",
                "correctness": "Parsing error",
                "completeness": "Parsing error"
            }
        }

In [ ]:

sample = evaluation_dataset[0]

sample_answer = knowledge_assistant(
    sample["question"],
    sample["context"]
)

sample_score = llm_judge(
    sample["question"],
    sample["context"],
    sample_answer,
    sample["ground_truth"]
)

print("Question:", sample["question"])
print("Assistant:", sample_answer)
print("\nJudge:")
print(json.dumps(sample_score, indent=2))

In [ ]:

results = {}

for item in evaluation_dataset:

    print(f"Evaluating {item['id']}/20...")

    answer = knowledge_assistant(
        item["question"],
        item["context"]
    )

    score = llm_judge(
        item["question"],
        item["context"],
        answer,
        item["ground_truth"]
    )

    results[item["id"]] = {
        "question": item["question"],
        "context": item["context"],
        "ground_truth": item["ground_truth"],
        "answer": answer,
        "scores": score
    }

print("\nEvaluation completed!")

In [ ]:

for test_id, result in results.items():

    scores = result["scores"]

    print("=" * 70)
    print(f"Test #{test_id}")
    print("Question:", result["question"])
    print("Answer:", result["answer"])

    print(
        f"Groundedness : {scores['groundedness']}/5"
    )

    print(
        f"Correctness  : {scores['correctness']}/5"
    )

    print(
        f"Completeness : {scores['completeness']}/5"
    )

In [ ]:

dimensions = [
    "groundedness",
    "correctness",
    "completeness"
]

aggregate_scores = {}

for dimension in dimensions:

    scores = [
        result["scores"][dimension]
        for result in results.values()
        if result["scores"][dimension] > 0
    ]

    aggregate_scores[dimension] = (
        sum(scores) / len(scores)
    )

print("\n===== AGGREGATE EVALUATION =====")

for dimension, score in aggregate_scores.items():

    print(
        f"{dimension.capitalize():15}: "
        f"{score:.2f}/5"
    )

lowest_dimension = min(
    aggregate_scores,
    key=aggregate_scores.get
)

highest_dimension = max(
    aggregate_scores,
    key=aggregate_scores.get
)

gap = (
    aggregate_scores[highest_dimension]
    - aggregate_scores[lowest_dimension]
)

print("\nLowest dimension:", lowest_dimension)
print("Highest dimension:", highest_dimension)
print(f"Gap: {gap:.2f} points")

In [ ]:

print("\n" + "=" * 60)
print("AI EVALUATION REPORT")
print("=" * 60)

for dimension, score in aggregate_scores.items():

    percentage = (score / 5) * 100

    print(
        f"{dimension.upper():15} "
        f"{score:.2f}/5 "
        f"({percentage:.1f}%)"
    )

print("-" * 60)

print(
    f"Lowest dimension: "
    f"{lowest_dimension.upper()}"
)

print(
    f"Lowest score: "
    f"{aggregate_scores[lowest_dimension]:.2f}/5"
)

print(
    f"Difference from highest: "
    f"{gap:.2f} points"
)

print("=" * 60)

In [ ]:

baseline = {
    "groundedness": round(
        aggregate_scores["groundedness"], 2
    ),
    "correctness": round(
        aggregate_scores["correctness"], 2
    ),
    "completeness": round(
        aggregate_scores["completeness"], 2
    )
}

print("Stored baseline:")
print(json.dumps(baseline, indent=2))

In [ ]:

def regression_test_runner(
    dataset,
    baseline,
    threshold=0.30
):

    print("\n")
    print("=" * 70)
    print("AI REGRESSION TEST")
    print("=" * 70)

    current_results = {}

    for item in dataset:

        answer = knowledge_assistant(
            item["question"],
            item["context"]
        )

        score = llm_judge(
            item["question"],
            item["context"],
            answer,
            item["ground_truth"]
        )

        current_results[item["id"]] = score

    current_scores = {}

    for dimension in dimensions:

        valid_scores = [
            result[dimension]
            for result in current_results.values()
            if result[dimension] > 0
        ]

        current_scores[dimension] = (
            sum(valid_scores) / len(valid_scores)
        )

    print("\nDimension Results")
    print("-" * 70)

    overall_pass = True

    for dimension in dimensions:

        baseline_score = baseline[dimension]
        current_score = current_scores[dimension]

        drop = baseline_score - current_score

        print(f"\nDimension: {dimension}")

        print(
            f"Baseline : {baseline_score:.2f}"
        )

        print(
            f"Current  : {current_score:.2f}"
        )

        print(
            f"Change   : {current_score - baseline_score:+.2f}"
        )

        if drop > threshold:

            print(
                f"FAIL ❌ — "
                f"{dimension} dropped by {drop:.2f}"
            )

            overall_pass = False

        else:

            print(
                f"PASS ✅ — "
                f"within threshold"
            )

    print("\n" + "=" * 70)

    if overall_pass:
        print("REGRESSION TEST: PASS ✅")
    else:
        print("REGRESSION TEST: FAIL ❌")

    print("=" * 70)

    return {
        "passed": overall_pass,
        "baseline": baseline,
        "current": current_scores,
        "results": current_results
    }

In [ ]:

regression_report = regression_test_runner(
    evaluation_dataset,
    baseline,
    threshold=0.30
)

In [ ]:

with open("evaluation_results.json", "w") as f:
    json.dump(
        results,
        f,
        indent=2
    )

with open("baseline.json", "w") as f:
    json.dump(
        baseline,
        f,
        indent=2
    )

print("Saved:")
print("evaluation_results.json")
print("baseline.json")

In [ ]:

limitations_document = """
# LLM-as-a-Judge Evaluation Limitations

## 1. Judge Models Can Be Wrong

An LLM judge is itself an AI model and can make mistakes.
It may incorrectly score an answer even when the answer is correct.

The judge can misunderstand:
- the question
- the context
- the ground truth
- the assistant answer

Therefore, judge scores should not automatically be treated as absolute truth.

## 2. Subjectivity

Some evaluation criteria are subjective.

For example, two evaluators may disagree about whether an answer
is "complete" or whether a small missing detail is important.

LLM judges can also have similar inconsistencies.

## 3. Position and Wording Bias

LLM judges may prefer:
- longer answers
- more detailed answers
- confident language
- particular wording

A shorter answer can sometimes be fully correct but receive a lower score.

## 4. Groundedness Limitations

The judge may fail to detect subtle hallucinations.

It may also incorrectly classify a logically inferred statement as
unsupported even when the inference is reasonable.

Groundedness evaluation works best when the supplied context is clear
and sufficiently detailed.

## 5. Correctness Limitations

Correctness is difficult when multiple answers can be valid.

For subjective questions, open-ended questions, creative tasks,
or questions with multiple acceptable solutions, a simple 1-5 score
may not be reliable.

## 6. Completeness Limitations

Completeness depends on what information the evaluator considers
important.

Different judges may disagree about whether an omitted detail matters.

## 7. Mathematical and Code Evaluation

LLM judges should not be the only evaluator for:
- exact mathematical calculations
- executable code
- security-sensitive code
- structured data validation
- exact numerical outputs

Deterministic automated tests are usually better for these cases.

## 8. Human Evaluation Is Still Necessary

Human evaluation should be used when:
- the task is subjective
- the answer affects important decisions
- safety is involved
- multiple answers can be valid
- subtle factual errors matter
- evaluation criteria are difficult to formalize
- the system is being validated for production

## 9. Best Practice

A strong production evaluation system should combine:

1. Automated deterministic tests
2. LLM-as-a-judge evaluation
3. Human evaluation
4. Regression testing
5. Real-world user feedback

LLM-as-a-judge should be treated as an evaluation signal,
not as an unquestionable source of truth.
"""

with open("EVALUATION_LIMITATIONS.md", "w") as f:
    f.write(limitations_document)

print("EVALUATION_LIMITATIONS.md created!")